Mar 14, 2025:

try getting wandb grid search working with current train.py code

In [2]:
cd /home/yang1641/PINNACLE/pinnacle

/home/yang1641/PINNACLE/pinnacle


In [3]:
# General
import numpy as np
import random
import argparse
import os
import copy

# Pytorch
import torch
import torch.nn as nn
from torch_geometric.utils.convert import to_networkx, to_scipy_sparse_matrix
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling

# Center loss
from center_loss import CenterLoss

# W&B
import wandb

# Own code
from generate_input import read_data, get_metapaths, get_centerloss_labels
import model as mdl
import utils
import minibatch_utils as mb_utils
from parse_args import get_args, get_hparams

# Seed
seed = 3
print("SEED:", seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) 
np.random.seed(seed)
random.seed(seed)
# torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# # Setup
# args = get_args()
# hparams_raw = get_hparams(args)

# save_log = args.save_prefix + "_gnn_train.log"
# save_graph = args.save_prefix + "_graph.pkl"
# save_model = args.save_prefix + "_model_save.pth"
# save_plots = args.save_prefix + "_train_embed_plots.pdf"
# save_ppi_embed = args.save_prefix + "_protein_embed.pth"
# save_mg_embed = args.save_prefix + "_mg_embed.pth"
# save_labels_dict = args.save_prefix + "_labels_dict.txt"



SEED: 3


In [6]:
parameters_dict = {
    'feat_mat':{
        'values':[1024, 2048]
        },
    'output': {
        'values': [4,8,16,32]
    }, 
    'lmbda': {
        'values': [0.1, 0.01, 0.001]
    }, 
    # learning rate for link prediction task
    # lr
    'lr': {
        'values': [0.01, 0.001]
    }, 
    # learning rate for protein label classification 
    # SOLVED: do we need this? - yes, because it's part of center loss
    # lr_cent
    'lr_cent': {
        'values': [0.1, 0.01, 0.001]
    }, 
    'n_heads': {
        'values': [4, 8]
    }, 
    # weight decay rate - optimal was 0.00001, according to methods
    'wd': {
        'values': [0.0001, 0.00001]
    }, 
    # drop out rate - will need to make it even fewer
    'dropout': {
        'values': [0.3, 0.4, 0.5, 0.6, 0.7]
    }, 
    # normalization layer - 
    # TODO: implement for later (in train.py and other code) if needed, as context ai concludes both layer norm and batch norm should be used
#     'norm': {
#         'values': ['bn', 'ln']
#     }, 
    # batch size - context ai did not include this as a hyperparameter
    # TODO: search for batch size as well if needed
#     'batch_size': {
#         # integers between 32 and 256
#         # with evenly-distributed logarithms 
#         'distribution': 'q_log_uniform',
#         'q': 1,
#         'min': math.log(32),
#         'max': math.log(256),
#       }
    }

In [7]:
parameters_dict = {
    'feat_mat':{
        'values':[1024, 2048]
        }
    }


sweep_config = {
    'method': 'grid'
    }

sweep_config['parameters'] = parameters_dict


from pprint import pprint

pprint(sweep_config)

{'method': 'grid', 'parameters': {'feat_mat': {'values': [1024, 2048]}}}


In [5]:
# Create the sweep
sweep_id = wandb.sweep(sweep_config, project="pytorch-sweeps-demo")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Create sweep with ID: o54z6ty4
Sweep URL: https://wandb.ai/yang1641-purdue-university/pytorch-sweeps-demo/sweeps/o54z6ty4


In [ ]:
def sweep_wrapper():
    # Start a W&B run for this hyperparam combo
    wandb.init()
    global hparams
    
    # Pull each hyperparam from wandb.config
    hparams["feat_mat"] = wandb.config.feat_mat
    hparams["lr"] = wandb.config.lr
    
    # Example: build model/optimizer using these hyperparams
    model = torch.nn.Linear(hparams["feat_mat"], 10)
    optimizer = torch.optim.Adam(model.parameters(), lr=hparams["lr"])
    center_loss = None  # or however you define it

    # Typical training loop
    for epoch in range(10):
        train(epoch, model, optimizer, center_loss)

# Run the sweep agent, which calls sweep_wrapper() for every combination
wandb.agent(sweep_id, function=sweep_wrapper)